# Resultados por película, género e idioma (incremental)

Procesa el lote `p_file_date`: combina las películas estrenadas desde 2000 con sus idiomas y géneros y carga el resultado en `movie_gold.results_movie_genre_language` con `MERGE`.

### Configuración

Carga las rutas y las funciones comunes, y lee `p_file_date` (fecha del lote).

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

### Lectura de fuentes Silver

Lee de `movie_silver` las tablas del lote (`file_date = p_file_date`): `movies`, `movies_languages` y `movie_genre`. Las dimensiones `languages` y `genres` se leen completas.

In [0]:
movies_df = (
    spark.read
    .table("movie_silver.movies")
    .filter(f"file_date = '{v_file_date}'")
    
)

In [0]:
languages_df = (
    spark.read
        .table("movie_silver.languages")
)

In [0]:
movie_languages_df = (
    spark.read
        .table("movie_silver.movies_languages")
        .filter(f"file_date = '{v_file_date}'")
)

In [0]:
genres_df = (
    spark.read
        .table("movie_silver.genres")
)

In [0]:
movies_genres_df = (
    spark.read
        .table("movie_silver.movie_genre")
        .filter(f"file_date = '{v_file_date}'")
)

In [0]:
from pyspark.sql.functions import col, desc, asc, lit

### Construcción del resultado

Filtra películas desde 2000, las une con idiomas y géneros y usa `p_file_date` como `created_date`.

In [0]:
results_movie_genre_language_df = (
    movies_df.filter(col("year_release_date") >= 2000)
        .withColumn("created_date", lit(v_file_date))
        .join(movie_languages_df, 
              movies_df["movie_id"] == movie_languages_df["movie_id"], "inner")
        .join(languages_df,
              movie_languages_df["language_id"] == languages_df["language_id"], "inner")
        .join(movies_genres_df, 
              movies_df["movie_id"] == movies_genres_df["movie_id"], "inner")
        .join(genres_df,
              movies_genres_df["genre_id"] == genres_df["genre_id"], "inner")
).select(
    movies_df["title"], movies_df["duration_time"], movies_df["release_date"], movies_df["vote_average"],
    languages_df["language_name"],
    genres_df["genre_name"],
    col("created_date")
).orderBy(
    col("release_date").asc()
)

### Escritura en Gold

Las celdas comentadas conservan la estrategia anterior (borrar partición + `append`). La carga activa usa `MERGE` más abajo.

In [0]:
#overwrite_partitioned_table("movie_gold","results_movie_genre_language","created_date", v_file_date)

In [0]:
"""
(
    results_movie_genre_language_df
    .write
        .mode("append")
        .partitionBy("created_date")
        .saveAsTable(f"movie_gold.results_movie_genre_language")
)
"""

## Utilizando Merge
Reconstruye el resultado incluyendo las llaves que necesita el `MERGE` y lo carga con `merge_delta_lake` (llave `movie_id` + `language_id` + `genre_id` + `created_date`), particionando por `created_date`.

In [0]:
results_movie_genre_language_df = (
    movies_df.filter(col("year_release_date") >= 2000)
        .withColumn("created_date", lit(v_file_date))
        .join(movie_languages_df, 
              movies_df["movie_id"] == movie_languages_df["movie_id"], "inner")
        .join(languages_df,
              movie_languages_df["language_id"] == languages_df["language_id"], "inner")
        .join(movies_genres_df, 
              movies_df["movie_id"] == movies_genres_df["movie_id"], "inner")
        .join(genres_df,
              movies_genres_df["genre_id"] == genres_df["genre_id"], "inner")
).select(
    movies_df["movie_id"], movies_df["title"], movies_df["duration_time"], movies_df["release_date"], movies_df["vote_average"],
    languages_df["language_name"], languages_df["language_id"],
    genres_df["genre_name"], genres_df["genre_id"],
    col("created_date")
).orderBy(
    col("release_date").asc()
)

In [0]:
merge_condition = 'trgt.movie_id = src.movie_id AND trgt.language_id = src.language_id AND trgt.genre_id = src.genre_id AND trgt.created_date = src.created_date'
merge_delta_lake(input_df = results_movie_genre_language_df, schema_name = "movie_gold", table_name = "results_movie_genre_language", merge_condition = merge_condition, partition_column = "created_date")

### Verificación

Cuenta registros por `created_date`.

In [0]:
%sql
SELECT created_date,
COUNT(*)
FROM movie_gold.results_movie_genre_language
GROUP BY created_date